# Monthly data pipeline (single notebook)
This notebook downloads/builds all required monthly features and writes one final panel CSV.

## 1) Setup

In [ ]:

import os
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data/raw"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "1960-01-01"
END_DATE = "2026-01-01"
START_YEAR = int(START_DATE[:4])



## 2) Futures prices and returns (corn, soybean, wheat)

In [ ]:

tickers = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
}

frames = []
for commodity, ticker in tickers.items():
    daily = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False,
    )

    if daily.empty:
        print(f"No data returned for {commodity} ({ticker}).")
        continue

    if isinstance(daily.columns, pd.MultiIndex):
        daily.columns = daily.columns.get_level_values(0)

    price_col = "Adj Close" if "Adj Close" in daily.columns else "Close"
    if price_col not in daily.columns:
        print(f"No price column available for {commodity} ({ticker}).")
        continue

    monthly = daily[[price_col]].resample("ME").last().copy()
    monthly["futures_return"] = np.log(monthly[price_col]).diff()
    monthly["commodity"] = commodity
    monthly = monthly.reset_index().rename(columns={"Date": "date", price_col: "futures_price"})
    frames.append(monthly)

if not frames:
    raise RuntimeError("No commodity data downloaded. Check internet connection and ticker symbols.")

futures_df = pd.concat(frames, ignore_index=True)
futures_df.to_csv(RAW_DIR / "futures_monthly.csv", index=False)
print("saved", RAW_DIR / "futures_monthly.csv", futures_df.shape)


## 3) Macro variables from FRED (USD index, interest rate, VIX)

In [ ]:

fred_api_key = os.getenv("FRED_API_KEY")
if not fred_api_key:
    # Fallback for Codespaces/Jupyter kernels that do not inherit shell exports
    env_file = PROJECT_ROOT / ".env"
    if env_file.exists():
        for raw_line in env_file.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            if key.strip() == "FRED_API_KEY":
                fred_api_key = value.strip().strip('"').strip("'")
                os.environ["FRED_API_KEY"] = fred_api_key
                break

if not fred_api_key:
    raise RuntimeError(
        "Set FRED_API_KEY (env var) or create PROJECT_ROOT/.env with FRED_API_KEY=<your_key>, then restart kernel."
    )

fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_map = {
    "usd_index": "DTWEXBGS",
    "interest_rate": "TB3MS",
    "vix": "VIXCLS",
}

macro_parts = []
for output_name, series_id in series_map.items():
    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json",
        "observation_start": START_DATE,
    }
    r = requests.get(fred_url, params=params, timeout=30)
    r.raise_for_status()
    obs = r.json().get("observations", [])

    df = pd.DataFrame(obs)
    if df.empty:
        df = pd.DataFrame(columns=["date", output_name])
    else:
        df = df[["date", "value"]].copy()
        df["date"] = pd.to_datetime(df["date"])
        df[output_name] = pd.to_numeric(df["value"], errors="coerce")
        df = df[["date", output_name]]

    if output_name in ["usd_index", "vix"]:
        df = df.set_index("date").resample("ME").mean().reset_index()
    else:
        df["date"] = pd.to_datetime(df["date"]).dt.to_period("M").dt.to_timestamp("M")

    macro_parts.append(df)

macro_df = macro_parts[0]
for part in macro_parts[1:]:
    macro_df = macro_df.merge(part, on="date", how="outer")
macro_df = macro_df.sort_values("date").reset_index(drop=True)
macro_df.to_csv(RAW_DIR / "macro_monthly.csv", index=False)
print("saved", RAW_DIR / "macro_monthly.csv", macro_df.shape)


## 4) ENSO + disaster/climate inputs

This block combines:
- ONI (NOAA CPC)
- FEMA monthly U.S. disaster declaration counts (all + selected hazard types)
- Optional local climate files if available (temperature/precipitation/drought/extreme heat)

In [ ]:

# --- 4a. ENSO ONI (NOAA CPC) ---
oni_url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
season_to_month = {
    "DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4,
    "AMJ": 5, "MJJ": 6, "JJA": 7, "JAS": 8,
    "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12,
}

oni_table = pd.read_csv(oni_url, sep=r"\s+")
oni_table.columns = [str(c).strip() for c in oni_table.columns]
required_oni_cols = {"SEAS", "YR", "ANOM"}
if not required_oni_cols.issubset(set(oni_table.columns)):
    raise ValueError(f"Unexpected ONI table columns: {oni_table.columns.tolist()}")

oni_table = oni_table.rename(columns={"SEAS": "season", "YR": "year", "ANOM": "enso_oni"})
oni_table["month"] = oni_table["season"].map(season_to_month)
oni_table = oni_table.dropna(subset=["month"])
oni_table["date"] = pd.to_datetime(
    dict(year=oni_table["year"].astype(int), month=oni_table["month"].astype(int), day=1)
) + pd.offsets.MonthEnd(0)
oni_table["enso_oni"] = pd.to_numeric(oni_table["enso_oni"], errors="coerce")
climate_df = oni_table[["date", "enso_oni"]].copy()

# --- 4b. FEMA disaster declarations (OpenFEMA API) ---
def fetch_fema_disasters(start_year: int = 2010, chunk_size: int = 5000) -> pd.DataFrame:
    base_url = "https://www.fema.gov/openfema-data-hub-api/disaster-declarations-summaries-v2"
    select_cols = ["declarationDate", "incidentType"]
    filters = f"declarationDate ge '{start_year}-01-01T00:00:00.000Z'"

    rows = []
    skip = 0
    while True:
        params = {
            "$select": ",".join(select_cols),
            "$filter": filters,
            "$top": chunk_size,
            "$skip": skip,
            "$orderby": "declarationDate asc",
        }
        resp = requests.get(base_url, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        batch = payload.get("DisasterDeclarationsSummaries", [])
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < chunk_size:
            break
        skip += chunk_size

    return pd.DataFrame(rows)

fema_raw = fetch_fema_disasters(start_year=START_YEAR)
if fema_raw.empty:
    print("No FEMA disaster records fetched; disaster features will be NA.")
    fema_monthly = pd.DataFrame(columns=["date", "disaster_event_count", "disaster_storm_count", "disaster_fire_count", "disaster_flood_count"])
else:
    fema_raw["date"] = pd.to_datetime(fema_raw["declarationDate"], errors="coerce").dt.to_period("M").dt.to_timestamp("M")
    fema_raw["incidentType"] = fema_raw["incidentType"].fillna("Unknown")

    monthly_total = fema_raw.groupby("date").size().rename("disaster_event_count").reset_index()

    def monthly_count_for(label: str, out_col: str) -> pd.DataFrame:
        m = (
            fema_raw.assign(is_type=fema_raw["incidentType"].str.lower().eq(label.lower()))
            .groupby("date")["is_type"]
            .sum()
            .rename(out_col)
            .reset_index()
        )
        return m

    storm = monthly_count_for("Severe Storm(s)", "disaster_storm_count")
    fire = monthly_count_for("Fire", "disaster_fire_count")
    flood = monthly_count_for("Flood", "disaster_flood_count")

    fema_monthly = monthly_total.merge(storm, on="date", how="left")                                 .merge(fire, on="date", how="left")                                 .merge(flood, on="date", how="left")

climate_df = climate_df.merge(fema_monthly, on="date", how="left")

disaster_cols = ["disaster_event_count", "disaster_storm_count", "disaster_fire_count", "disaster_flood_count"]
for col in disaster_cols:
    if col in climate_df.columns:
        climate_df[col] = pd.to_numeric(climate_df[col], errors="coerce").fillna(0.0)

# --- 4c. Optional local monthly climate files ---
optional_files = [
    (RAW_DIR / "drought_monthly_input.csv", "drought_index"),
    (RAW_DIR / "temp_anomaly_input.csv", "temperature_anomaly"),
    (RAW_DIR / "precip_anomaly_input.csv", "precipitation_anomaly"),
    (RAW_DIR / "extreme_heat_input.csv", "extreme_heat_events"),
]

for file_path, col_name in optional_files:
    if file_path.exists():
        temp_df = pd.read_csv(file_path)
        if "date" not in temp_df.columns or col_name not in temp_df.columns:
            raise ValueError(f"{file_path} must include columns date and {col_name}")
        temp_df = temp_df[["date", col_name]].copy()
        temp_df["date"] = pd.to_datetime(temp_df["date"]).dt.to_period("M").dt.to_timestamp("M")
        climate_df = climate_df.merge(temp_df, on="date", how="outer")
    else:
        print(f"missing optional file: {file_path}; column {col_name} will be mostly NA")

climate_df = climate_df.sort_values("date").reset_index(drop=True)
climate_df.to_csv(RAW_DIR / "climate_disaster_monthly.csv", index=False)
print("saved", RAW_DIR / "climate_disaster_monthly.csv", climate_df.shape)
climate_df.tail()


## 5) Merge all monthly blocks into final panel

In [ ]:
panel = futures_df.merge(macro_df, on="date", how="left")
panel = panel.merge(climate_df, on="date", how="left")

required_cols = [
    "futures_return",
    "enso_oni",
    "temperature_anomaly",
    "precipitation_anomaly",
    "drought_index",
    "extreme_heat_events",
    "disaster_event_count",
    "disaster_storm_count",
    "disaster_fire_count",
    "disaster_flood_count",
    "usd_index",
    "interest_rate",
    "vix",
]
for c in required_cols:
    if c not in panel.columns:
        panel[c] = pd.NA

panel = panel.sort_values(["commodity", "date"]).reset_index(drop=True)

core_required = ["futures_price", "usd_index", "interest_rate", "vix", "enso_oni"]
first_valid_date = panel.dropna(subset=core_required)["date"].min()
if pd.notna(first_valid_date):
    panel = panel[panel["date"] >= first_valid_date].copy()

panel.to_csv(PROCESSED_DIR / "monthly_panel.csv", index=False)

print(panel.head())
print("\nMissing values by required variable:")
print(panel[required_cols].isna().sum())
print("\nSaved", PROCESSED_DIR / "monthly_panel.csv", panel.shape)
